# ICU-SEWS · v2 + Platt 최종 학습과 test 평가
런타임 → 런타임 유형 변경 → **GPU(T4 등)** 를 선택하고 위에서부터 실행하세요.

- 기존 `SEWS_GRU/prepared` 입력을 재사용합니다. v2/v3 결과 폴더를 덮어쓰지 않습니다.
- Python **3.10**, v2 설정: GRU 64 / 1층 / dropout 0.2 / lr 0.0003 / 양성 가중치 / batch 128.
- **3 epochs 고정**: 기존 v2 최적 epoch `[2,2,3,3,3]`의 중앙값. 별도 early stopping 없음.
- 기존 dev fold 0~3: GRU 학습·전처리 통계 산출. fold 4: 병원×패혈증 여부를 층화해 환자 절반씩 **Platt 학습 / 임계값 설정**.
- 전체 환자 기준 약 **68% / 8.5% / 8.5% / test 15%**. test 환자 분할은 바꾸지 않습니다.
- 비패혈증 환자 중 경보를 한 번 이상 받는 비율 **10% 목표**로 단일 확률 임계값을 결정합니다. 확률 임계값 0.10을 뜻하지 않습니다.
- 모델·보정기·임계값을 고정한 뒤 test를 평가합니다. test 결과로 설정을 바꾸거나 다른 실행을 골라 보고하지 마세요.
- 매 epoch Drive 저장 / 재개 지원. 완료한 단계는 다시 계산하지 않고 결과를 재사용합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
from datetime import datetime
from zoneinfo import ZoneInfo

SOURCE_ROOT = Path('/content/drive/MyDrive/capston')
WORK_ROOT = Path('/content/drive/MyDrive/SEWS_GRU')
RUN_NAME = 'gru_v2_platt_final'
OUT = WORK_ROOT / 'final' / RUN_NAME
PIN = WORK_ROOT / 'final_code_pins' / f'{RUN_NAME}.txt'
LOG_ROOT = WORK_ROOT / 'final_logs' / RUN_NAME
print('최종 결과:', OUT)

## 1. 코드와 Python 3.10 GPU 환경
첫 실행의 Git 코드 버전을 Drive에 저장하고 재연결 후에도 같은 버전을 사용합니다.
기존 학습·비교용 저장소와 별도 폴더를 사용합니다. 로그에는 내부 오류도 표시됩니다.

In [ ]:
import subprocess
import sys
import shutil
import os

os.environ['MPLBACKEND'] = 'Agg'
REPO_URL = 'https://github.com/sogang-icu-sews/ICU-SEWS.git'
BRANCH = 'codex/gru-colab'
REPO = Path('/content/ICU-SEWS-GRU-final')
def run(command, log_path=None):
    from collections import deque
    from contextlib import nullcontext
    tail = deque(maxlen=80)
    if log_path is not None:
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)
    log_context = log_path.open("w", encoding="utf-8") if log_path is not None else nullcontext(None)
    with log_context as log:
        with subprocess.Popen([str(x) for x in command], stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace") as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                tail.append(line)
                if log is not None:
                    log.write(line)
                    log.flush()
            returncode = process.wait()
    if returncode:
        details = "".join(tail)
        raise RuntimeError(f"실행 실패 (종료 코드 {returncode}). 로그: {log_path}\n마지막 출력:\n{details}")
    return subprocess.CompletedProcess([str(x) for x in command], returncode)


if not REPO.exists():
    run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, REPO])
origin = subprocess.check_output(['git', '-C', str(REPO), 'remote', 'get-url', 'origin'], text=True).strip()
assert origin == REPO_URL, '저장소 주소가 다릅니다.'
if PIN.exists():
    commit = PIN.read_text().strip()
    assert len(commit) == 40 and all(c in '0123456789abcdef' for c in commit)
    run(['git', '-C', REPO, 'checkout', '--detach', commit])
else:
    assert not OUT.exists(), '결과 폴더가 있으나 코드 버전 기록이 없습니다. 기존 실행을 먼저 확인하세요.'
    run(['git', '-C', REPO, 'fetch', 'origin', BRANCH])
    commit = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', f'origin/{BRANCH}'], text=True).strip()
    run(['git', '-C', REPO, 'checkout', '--detach', commit])
    assert (REPO / 'scripts/final_gru.py').is_file(), '최종 학습 코드가 있는 버전이 필요합니다.'
    PIN.parent.mkdir(parents=True, exist_ok=True)
    PIN.write_text(commit + '\n')
print('고정 코드 버전:', commit)
run(['nvidia-smi'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
UV = [sys.executable, '-m', 'uv']
PY = Path('/content/sews-final-py310/bin/python')
if not PY.exists():
    run(UV + ['venv', '--python', '3.10', str(PY.parent.parent)])
run(UV + ['pip', 'install', '--python', str(PY), 'torch==2.8.0',
          '--index-url', 'https://download.pytorch.org/whl/cu126'])
run(UV + ['pip', 'install', '--python', str(PY), '-r', str(REPO / 'requirements-gru-eval.txt')])
run([PY, '-c', "import sys,torch; print(sys.version); print(torch.__version__); "
     "assert sys.version_info[:2] == (3,10); assert torch.cuda.is_available(), 'GPU 런타임을 선택하세요'; "
     "print(torch.cuda.get_device_name(0))"])

## 2. 입력 준비
Drive의 `SEWS_GRU/prepared`를 Colab 임시 디스크로 복사합니다.
이미 준비된 파일이 있으면 원본 ZIP을 다시 처리하지 않습니다.
없을 때만 `capston/training.zip`에서 생성합니다. 스케일링 통계는 다음 학습 단계에서
**최종 GRU 학습 환자만 사용해 새로 산출**합니다. 기존 `final_dev` 파라미터를 쓰지 않습니다.

In [ ]:
import json

LOCAL_ROOT = Path("/content/sews_gru_final_inputs")
CACHED_ROOT = WORK_ROOT / "prepared"
data_files = ["seq_values.npy", "seq_row_start.npy", "eval_meta.parquet"]
art_files = ["preprocess_config.json", "feature_registry.json", "seq_meta.json", "patient_split.csv"]
param_files = [f"preprocess_params/fold{k}.json" for k in range(5)]

def ready(root):
    static_ok = any((root / "data" / p).is_file() for p in ["gru_static.parquet", "features.parquet"])
    return static_ok and all((root / "data" / p).is_file() for p in data_files) and all(
        (root / "artifacts" / p).is_file() for p in art_files + param_files)

def copy_inputs(source, destination):
    static = "gru_static.parquet" if (source / "data/gru_static.parquet").exists() else "features.parquet"
    files = [("data", p) for p in data_files + [static]]
    files += [("artifacts", p) for p in art_files + param_files]
    marker = source / "artifacts/gru_preparation_complete.json"
    if marker.exists():
        files.append(("artifacts", "gru_preparation_complete.json"))
    provenance = source / "artifacts/gru_source_files.csv"
    if provenance.exists():
        files.append(("artifacts", "gru_source_files.csv"))
    for folder, name in files:
        target = destination / folder / name
        target.parent.mkdir(parents=True, exist_ok=True)
        temp = target.with_name(target.name + ".partial")
        shutil.copyfile(source / folder / name, temp)
        temp.replace(target)

if not ready(LOCAL_ROOT):
    prepared = next((p for p in [CACHED_ROOT, SOURCE_ROOT] if ready(p)), None)
    if prepared is not None:
        copy_inputs(prepared, LOCAL_ROOT)
    else:
        source_zip = SOURCE_ROOT / "training.zip"
        assert source_zip.is_file(), f"전처리 파일 또는 {source_zip}이 필요합니다."
        local_zip = Path("/content/sews_training.zip")
        shutil.copyfile(source_zip, local_zip)
        run([PY, REPO / "scripts/prepare_gru_colab.py", "--archive", local_zip, "--out", LOCAL_ROOT])
        copy_inputs(LOCAL_ROOT, CACHED_ROOT)
assert ready(LOCAL_ROOT)
print("학습 입력 준비 완료:", LOCAL_ROOT)

## 3. 최종 GRU 학습 → Platt 보정 → 경보 임계값 확정
3 epochs 학습 후 Platt 보정은 가중치 없이 학습합니다. 확률 순위가 뒤집히는 보정 결과면
test에 진입하지 않고 중단합니다.

학습이 끊기면 같은 RUN_NAME으로 준비 셀부터 다시 실행하세요. 완료 epoch부터 재개합니다.
`frozen.json`이 있으면 모델·보정기·임계값이 이미 확정된 실행이므로 학습을 건너뜁니다.

In [ ]:
stamp = datetime.now(ZoneInfo('Asia/Seoul')).strftime('%Y%m%d_%H%M%S_%f')
run([PY, '-u', REPO / 'scripts/final_gru.py', 'fit',
     '--root', LOCAL_ROOT, '--out', OUT, '--device', 'cuda', '--resume'],
    log_path=LOG_ROOT / f'fit_{stamp}.log')

development = json.loads((OUT / 'development_summary.json').read_text())
print('환자 수:', development['patients'])
print('확정 임계값:', development['threshold'])
print('임계값 설정 그룹의 실제 환자 오경보율:',
      development['threshold_fitting_scores']['nonsepsis_patient_false_alarm_rate'])
print('위 값은 임계값을 맞춘 그룹의 점수입니다. 최종 성능은 다음 test에서 확인합니다.')

## 4. 고정된 모델로 test 평가
이 셀은 실제 test 15%의 성능을 확인합니다. 학습·보정·임계값 변경은 하지 않습니다.
중단 시 동일한 고정 모델로 재실행할 수 있습니다. 완료된 경우 저장된 결과를 확인합니다.
이후 test 성능에 맞춰 설정을 조정하면 더 이상 독립적인 최종 평가가 아닙니다.

In [ ]:
stamp = datetime.now(ZoneInfo('Asia/Seoul')).strftime('%Y%m%d_%H%M%S_%f')
run([PY, '-u', REPO / 'scripts/final_gru.py', 'evaluate',
     '--root', LOCAL_ROOT, '--out', OUT, '--device', 'cuda'],
    log_path=LOG_ROOT / f'test_{stamp}.log')

## 5. 최종 결과 확인
결과만 다시 볼 때는 Drive 연결 셀과 이 셀부터 실행하면 됩니다. GPU 환경 설치나 재학습은 필요 없습니다.
확률 지표는 전체·병원별·60시간 이내, 경보 지표는 전체 기록 및 병원별로 보고합니다.
같은 고정 임계값을 모든 환자에게 적용하며 test에서 10% 오경보율을 보장하지 않습니다.

조기 감지 시각은 첫 양성 라벨 +6시간으로 추정합니다. 첫 행부터 양성인 환자는 조기 감지·선행 시간에서만 제외합니다.
매시간 경보 플래그를 평가하며, 경보 억제 시간은 없습니다. 경보 시간 수는 알림 발생 횟수가 아닙니다.
이 보고서는 단일 최종 모델의 점 추정치이며 신뢰구간은 포함하지 않습니다.

In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

summary = json.loads((OUT / 'test_summary.json').read_text())
assert summary['complete'] and summary['test_evaluated']
print('읽는 결과:', OUT)
print('Test 환자 수:', summary['patients'], '/ 시간 행 수:', summary['rows'])
print('고정 경보 임계값:', summary['threshold']['threshold'])

print('① 최종 test 확률 성능')
prob = pd.read_csv(OUT / 'test_probability_metrics.csv')
display(prob.round(5))

print('② 최종 test 경보 성능 — 모든 그룹에 동일한 임계값')
alarms = pd.read_csv(OUT / 'test_alarm_metrics.csv')
cols = ['scope', 'row_sensitivity', 'row_PPV', 'row_specificity',
        'nonsepsis_patient_false_alarm_rate', 'early_recall_6to12h',
        'pre_onset_recall_0to12h', 'median_lead_hours_detected', 'Utility']
display(alarms[cols].round(5))

print('③ 실제 경보 부담·조기 평가 대상 수')
display(alarms[['scope', 'alarm_hours_per100h', 'false_alarm_hours_per100_nonsepsis_h',
                'timing_eligible_patients', 'timing_excluded_starts_positive']].round(4))
display(Image(filename=str(OUT / 'test_calibration.png')))
display(Image(filename=str(OUT / 'test_calibration_zoom.png')))
print('공유할 파일: test_probability_metrics.csv, test_alarm_metrics.csv, test_summary.json')

## 저장 파일과 다음 연결
- `final.pt`: 3 epochs 최종 GRU, 구조, 전처리 설정·통계, 입력 순서.
- `platt.json`: Platt 계수와 적용 공식. `threshold.json`: 보정 확률의 단일 경보 임계값.
- `preprocessing.json`, `patient_roles.csv`, `history.json`: 전처리·환자 역할·학습 기록.
- `frozen.json`: test 전에 확정된 파일의 해시. 확정 이후 변경된 파일은 거부합니다.
- `test_predictions.parquet`: test 시간별 원래 확률·보정 확률·경보 플래그.
- `test_*metrics.csv`, `test_summary.json`, `test_calibration*.png`: 최종 test 결과.

이전 비교 노트북의 fold별 보정기·임계값을 여기에 섞지 마세요. API 연결 시 최종 GRU와
그 GRU에 맞춘 전처리·Platt·임계값을 한 묶음으로 사용해야 합니다. API 연결은 이 노트북에서 수행하지 않습니다.